In [31]:
import pandas as pd
import numpy as np
#import dask.DataFrame as dd
%matplotlib inline
import seaborn as sns
import json

#pd.set_option('display.max_colwidth', -1)
#pd.set_option('display.max_rows', -1)

In [2]:
fname = 'trip_context_data_v3.csv'
first_groupby='type'
second_groupby='ContextType'
third_groupby='triggerOrContextName'
fnam_prefix = fname.split('.')[0]

In [3]:
dataframe = pd.read_csv(fname)

In [4]:
new_df = pd.DataFrame(columns=['grouped_by', 'column_name', 'null_pct', 'min','mean','std', 'max'])
new_df['grouped_by'] = ['timeStamp' for x in range(len(dataframe.columns))]
new_df['column_name'] = dataframe.columns.tolist()


In [5]:
numericalCols = dataframe.select_dtypes(include=[np.number]).columns.tolist()
catCols = list(set(dataframe.columns) -set(numericalCols))


In [6]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Columns: 745 entries, VIN to type
dtypes: float64(711), object(34)
memory usage: 2.8+ MB


## Find missing Values and their percent

In [7]:
missing_dataframe = dataframe.isna()

In [8]:
dataframe_missing_count = missing_dataframe.sum()

In [9]:
print("NULL percent is files with context == Trip")
dataframe_missing_pct = dataframe_missing_count/len(dataframe)
dataframe_missing_pct

NULL percent is files with context == Trip


VIN                         0.000000
DeviceValue                 1.000000
contexts                    1.000000
ContextType                 0.000000
TimeStampStartContext       1.000000
                              ...   
TotalDistanceEndTrip        0.895277
TotalDistanceUnitEndTrip    0.895277
timestamp                   0.000000
triggerOrContextName        0.100616
type                        0.010267
Length: 745, dtype: float64

In [10]:
zero_df = (dataframe == 0).sum(axis=0)
zero_df_pct = zero_df/len(dataframe)
zero_df_pct

VIN                         0.0
DeviceValue                 0.0
contexts                    0.0
ContextType                 0.0
TimeStampStartContext       0.0
                           ... 
TotalDistanceEndTrip        0.0
TotalDistanceUnitEndTrip    0.0
timestamp                   0.0
triggerOrContextName        0.0
type                        0.0
Length: 745, dtype: float64

In [11]:
min_df = dataframe.min()
min_df

VIN                                   HNV37-550052
DeviceValue                                    NaN
contexts                                       NaN
ContextType                                   Trip
TimeStampStartContext                          NaN
                                      ...         
TotalDistanceStartTrip                           7
TotalDistanceUnitStartTrip                      64
TotalDistanceEndTrip                             7
TotalDistanceUnitEndTrip                        64
timestamp                     2020-02-03T09:45:34Z
Length: 714, dtype: object

In [12]:
max_df = dataframe.max()
max_df

VIN                                     ZE1-092275
DeviceValue                                    NaN
contexts                                       NaN
ContextType                                   Trip
TimeStampStartContext                          NaN
                                      ...         
TotalDistanceStartTrip                       11332
TotalDistanceUnitStartTrip                      64
TotalDistanceEndTrip                         11364
TotalDistanceUnitEndTrip                        64
timestamp                     2020-02-05T23:55:00Z
Length: 714, dtype: object

In [13]:
mean_df = dataframe.mean()
mean_df

DeviceValue                           NaN
contexts                              NaN
TimeStampStartContext                 NaN
TimeStampEndContext                   NaN
DurationTripDay               4895.775000
                                 ...     
LocationSourceAtEnd              0.862745
TotalDistanceStartTrip        2403.019608
TotalDistanceUnitStartTrip      64.000000
TotalDistanceEndTrip          2407.254902
TotalDistanceUnitEndTrip        64.000000
Length: 711, dtype: float64

In [14]:
std_df = dataframe.std()
std_df

DeviceValue                           NaN
contexts                              NaN
TimeStampStartContext                 NaN
TimeStampEndContext                   NaN
DurationTripDay               4997.424539
                                 ...     
LocationSourceAtEnd              0.347540
TotalDistanceStartTrip        2456.104415
TotalDistanceUnitStartTrip       0.000000
TotalDistanceEndTrip          2459.021040
TotalDistanceUnitEndTrip         0.000000
Length: 711, dtype: float64

In [15]:
max_df = dataframe.max()
max_df

VIN                                     ZE1-092275
DeviceValue                                    NaN
contexts                                       NaN
ContextType                                   Trip
TimeStampStartContext                          NaN
                                      ...         
TotalDistanceStartTrip                       11332
TotalDistanceUnitStartTrip                      64
TotalDistanceEndTrip                         11364
TotalDistanceUnitEndTrip                        64
timestamp                     2020-02-05T23:55:00Z
Length: 714, dtype: object

In [16]:
min_df

VIN                                   HNV37-550052
DeviceValue                                    NaN
contexts                                       NaN
ContextType                                   Trip
TimeStampStartContext                          NaN
                                      ...         
TotalDistanceStartTrip                           7
TotalDistanceUnitStartTrip                      64
TotalDistanceEndTrip                             7
TotalDistanceUnitEndTrip                        64
timestamp                     2020-02-03T09:45:34Z
Length: 714, dtype: object

In [17]:
#res_df=pd.concat([new_df, missing_dataframe,min_df, mean_df, std_df, max_df])
new_df['null_pct'] = dataframe_missing_pct.tolist()
def default_retrieve(df, col, default='test'):
    try:
        res = df[col]
    except KeyError:
        res = default
    return res

min_vals, max_vals, std_vals, mean_vals = list(), list(), list(), list()
for col in new_df.column_name:
    min_vals.append(default_retrieve(min_df, col, 'dummy_min'))
    max_vals.append(default_retrieve(max_df, col, 'dummy_max'))
    std_vals.append(default_retrieve(std_df, col, 'dummy_std'))
    mean_vals.append(default_retrieve(mean_df, col, 'dummy_mean'))


new_df['min'] = min_vals
new_df['max'] = max_vals
new_df['std'] = std_vals
new_df['mean'] = mean_vals
new_df['zeros_pct'] = zero_df_pct.values.tolist()
out_fname =  fnam_prefix + '_summary.csv'
new_df.to_csv(out_fname, index=False)


In [33]:
all_null_cols = filter(lambda x: all(dataframe[x].isna()), dataframe.columns)
with open('all_null_columns.json', 'w+') as fd:
    json.dump(all_null_cols, fd)
#print(list(all_null_cols))

TypeError: Object of type filter is not JSON serializable

## Find descriptive stats grouped by categorical field values

In [19]:
grouped_stats = dict()

In [20]:
dataframe_trigger_grouped = dataframe.groupby(third_groupby)
dataframe_na_vals_cnt_grouped = dataframe_trigger_grouped.apply(lambda x: pd.isna(x)).count()

grouped_stats[third_groupby] = dataframe_na_vals_cnt_grouped/dataframe.groupby(third_groupby).count()

In [21]:
grouped_stats

{'triggerOrContextName':                                                     ACC_DrivingTimeInTrip  \
 triggerOrContextName                                                        
 Trip                                                            87.600000   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...               9.521739   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                    inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                    inf   
 
                                                     ACElectricityConsumption  \
 triggerOrContextName                                                           
 Trip                                                                     inf   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...                 33.692308   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                       inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                       inf   
 
                  

In [22]:
dataframe_context_grouped = dataframe.groupby(second_groupby)
dataframe_na_vals_context_cnt_grouped = dataframe_context_grouped.apply(lambda x: pd.isna(x)).count()

grouped_stats[second_groupby] = dataframe_na_vals_context_cnt_grouped/dataframe.groupby(second_groupby).count()

In [23]:
grouped_stats

{'triggerOrContextName':                                                     ACC_DrivingTimeInTrip  \
 triggerOrContextName                                                        
 Trip                                                            87.600000   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...               9.521739   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                    inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                    inf   
 
                                                     ACElectricityConsumption  \
 triggerOrContextName                                                           
 Trip                                                                     inf   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...                 33.692308   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                       inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                       inf   
 
                  

In [24]:
dataframe_type_grouped = dataframe.groupby(first_groupby)
dataframe_na_vals_type_cnt_grouped = dataframe_type_grouped.apply(lambda x: pd.isna(x)).count()

grouped_stats[first_groupby] = dataframe_na_vals_type_cnt_grouped/dataframe.groupby(first_groupby).count()

In [25]:
grouped_stats

{'triggerOrContextName':                                                     ACC_DrivingTimeInTrip  \
 triggerOrContextName                                                        
 Trip                                                            87.600000   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...               9.521739   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                    inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                    inf   
 
                                                     ACElectricityConsumption  \
 triggerOrContextName                                                           
 Trip                                                                     inf   
 UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPe...                 33.692308   
 UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit                       inf   
 UCD/Sent/Trigger/Trg_periodic_30sec_Probe_Drive...                       inf   
 
                  

In [26]:
for k,v in grouped_stats.items():
    grouped_stats[k].to_csv('%s_%s'%(fnam_prefix, 'grouped_by_' + str(k) + '.csv'), index=False)

In [27]:
grouped_stats['triggerOrContextName']

,ACC_DrivingTimeInTrip,ACElectricityConsumption,AcceleratorPedalOpeningRatio,AirTempDrivingStartupUnitEV,AirTempDrivingStartupUnitICE,AirTempDrivingStartupValueEV,AirTempDrivingStartupValueICE,AirTempStartingTimeUnit,AirTempStartingTimeValue,AtIdlingACElectricityConsumption,...,VCMEVTractionMotorRegenerationDay,VCMEVTractionMotorRegenerationLife,VCMEVTractionMotorRegenerationMonth,VCMEVTractionMotorRegenerationYear,VIN,VehicleLocationGNSS_Timestamp,contexts,timestamp,triggerOrContextName,type
triggerOrContextName,,,,,,,,,,,,,,,,,,,,,
Trip,87.600000,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,87.600000,inf,inf,87.600000,NaN,inf
UCD/Sent/Trigger/Trg_IGN_OFF_Probe_DriverInfoPermit,9.521739,33.692308,inf,10.428571,10.428571,10.428571,10.428571,inf,inf,33.692308,...,inf,inf,inf,inf,8.588235,inf,inf,8.588235,NaN,8.588235
UCD/Sent/Trigger/Trg_IGN_ON_Probe_DriverInfoPermit,inf,inf,inf,inf,inf,inf,inf,24.333333,24.333333,inf,...,inf,inf,inf,inf,7.684211,inf,inf,7.684211,NaN,7.684211
UCD/Sent/Trigger/Trg_periodic_30sec_Probe_DriverInfoPermit,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,1.347692,inf,inf,1.347692,NaN,1.347692
